In [ ]:

import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

n_particles = 20
n_features = X.shape[1]
max_iter = 100
w = 0.7
c1 = 1.5
c2 = 1.5

positions = np.random.randint(0, 2, (n_particles, n_features))
velocities = np.random.uniform(-1, 1, (n_particles, n_features))

def fitness(position):
    selected = position.astype(bool)

    if np.sum(selected) == 0:
        return 1.0

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000)
    )

    model.fit(X_train[:, selected], y_train)
    predictions = model.predict(X_test[:, selected])
    accuracy = accuracy_score(y_test, predictions)

    return 1 - accuracy + 0.01 * np.mean(selected)

pbest = positions.copy()
pbest_scores = np.array([fitness(p) for p in positions])

gbest_index = np.argmin(pbest_scores)
gbest = pbest[gbest_index].copy()
gbest_score = pbest_scores[gbest_index]

for iteration in range(max_iter):
    for i in range(n_particles):
        r1 = np.random.rand(n_features)
        r2 = np.random.rand(n_features)

        velocities[i] = (
            w * velocities[i]
            + c1 * r1 * (pbest[i] - positions[i])
            + c2 * r2 * (gbest - positions[i])
        )

        probabilities = 1 / (1 + np.exp(-np.clip(velocities[i], -500, 500)))
        positions[i] = (np.random.rand(n_features) < probabilities).astype(int)

        score = fitness(positions[i])

        if score < pbest_scores[i]:
            pbest[i] = positions[i].copy()
            pbest_scores[i] = score

    gbest_index = np.argmin(pbest_scores)

    if pbest_scores[gbest_index] < gbest_score:
        gbest = pbest[gbest_index].copy()
        gbest_score = pbest_scores[gbest_index]

selected_features = feature_names[gbest.astype(bool)]

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

model.fit(X_train[:, gbest.astype(bool)], y_train)
predictions = model.predict(X_test[:, gbest.astype(bool)])
accuracy = accuracy_score(y_test, predictions)

print("Selected Features:")
print(selected_features)

print("Number of Selected Features:", len(selected_features))
print("Total Features:", n_features)
print("Classification Accuracy:", round(accuracy * 100, 2), "%")


Selected Features:
['mean texture' 'mean smoothness' 'mean concave points' 'radius error'
 'perimeter error' 'area error' 'smoothness error' 'compactness error'
 'concave points error' 'worst area' 'worst concavity' 'worst symmetry']
Number of Selected Features: 12
Total Features: 30
Classification Accuracy: 99.12 %
